# Chatbot Example


In this lesson, you will familiarize yourself with the chatbot example you will work on during this chapter(Chapter 4). The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.


## Import Libraries


In [ ]:
!pip install openai arxiv python-dotenv -q

In [17]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv

## Tool Functions


In [18]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  


In [19]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    client = arxiv.Client()

    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    print(f"Saving papers information in: {path}")
    
    file_path = os.path.join(path, "papers_info.json")

    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids


In [6]:
search_papers("computers")


Results are saved in: papers\computers\papers_info.json


['1312.3300v1', '2207.05241v1', '2603.19778v1', '2601.11095v1', '2012.10468v1']

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.


In [20]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."


In [9]:
extract_info('1312.3300v1')


'{\n  "title": "Numerical Reproducibility and Parallel Computations: Issues for Interval Algorithms",\n  "authors": [\n    "Nathalie Revol",\n    "Philippe Th\\u00e9veny"\n  ],\n  "summary": "What is called \\"numerical reproducibility\\" is the problem of getting the same result when the scientific computation is run several times, either on the same machine or on different machines, with different types and numbers of processing units, execution environments, computational loads etc. This problem is especially stringent for HPC numerical simulations. In what follows, the focus is on parallel implementations of interval arithmetic using floating-point arithmetic. For interval computations, numerical reproducibility is of course an issue for testing and debugging purposes. However, as long as the computed result encloses the exact and unknown result, the inclusion property, which is the main property of interval arithmetic, is satisfied and getting bit for bit identical results may not

## Tool Schema


Here are the schemas of each tool in **OpenAI function-calling format**.


In [21]:
# OpenAI uses a slightly different schema format compared to Anthropic.
# Tools are wrapped under a 'function' key with 'type': 'function'.
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search for papers on arXiv based on a topic and store their information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search for"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to retrieve",
                        "default": 5
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_info",
            "description": "Search for information about a specific paper across all topic directories.",
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": "The ID of the paper to look for"
                    }
                },
                "required": ["paper_id"]
            }
        }
    }
]


## Tool Mapping


This code handles tool mapping and execution.


In [22]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
    elif isinstance(result, list):
        result = ', '.join(result)
    elif isinstance(result, dict):
        result = json.dumps(result, indent=2)
    else:
        result = str(result)
    return result


## Chatbot Code


The chatbot handles the user's queries one by one, but it does not persist memory across the queries.


In [23]:
import aisuite as ai

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")

client = ai.Client(
    {
        "openai": {
            "api_key": openai_api_key,
            "base_url": openai_base_url,
        }
    }
)


### Query Processing


In [35]:
def process_query(query):
    messages = [{'role': 'user', 'content': query}]

    response = client.chat.completions.create(
        model='openai:gpt-4o',
        #model='openai:gpt-5-nano',
        tools=tools,
        messages=messages
    )

    process_flag = True
    while process_flag:
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # If no tool calls, just print the text and stop
        #if finish_reason == 'stop' or not message.tool_calls:
        if not message.tool_calls:
            print(message.content)
            process_flag = False

        # Handle tool calls
        elif finish_reason == 'tool_calls':
            # Add assistant message (with tool calls) to history
            messages.append(message)

            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                tool_call_id = tool_call.id

                print(f"Calling tool {tool_name} with args {tool_args}")
                result = execute_tool(tool_name, tool_args)

                # Append tool result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": result
                })

            # Send updated messages back to model
            response = client.chat.completions.create(
                model='openai:gpt-4o',
                #model='openai:gpt-5-nano',
                tools=tools,
                messages=messages
            )

            # If next response is just text, print and stop
            next_message = response.choices[0].message
            if response.choices[0].finish_reason == 'stop':
                print(next_message.content)
                process_flag = False


### Chat Loop


In [36]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")


Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"


In [38]:
chat_loop()


Type your queries or 'quit' to exit.
Here are two notable papers on the topic of interpretability in large language models (LLMs):

1. **"Towards Interpretable and Trustworthy Functional MRI Analysis via Self-Supervised Learning"**
   - Authors: Jiayu Chen, Shuchin Aeron, and others
   - Summary: This paper explores the use of self-supervised learning techniques to improve the interpretability of models analyzing functional MRI data. While not solely focused on LLMs, the methodologies discussed can offer insights into interpretability techniques useful for improving trust in complex models, including LLMs.

2. **"Interpretability of Deep Learning Models: A Survey of Results"**
   - Authors: Nidhi Gupta, Neeraj Dhanraj Bokde
   - Summary: This survey paper reviews various methods and approaches for interpretability in deep learning models. It discusses techniques that can be applied across different model types, including LLMs, to make their decision-making processes more transparent an